# MB1 v0.2 — Boundary-Rich Semantic Moment Candidates (Mode A only)

This notebook prepares model-free raw-video evidence for later GPT-5.6 Sol QC and annotation. It assigns no semantic labels, uses no previous interval GT, and does not run CLIP, VLM, optical flow, tracking, or any other model.

Required Kaggle inputs: raw AIC dataset, MB1 v0.1 candidate pack, MB1 AI-QC annotations, and the RT2 AI benchmark bundle. Repository cloning is the only operation that can require Internet; the experiment and all assets run offline. Output ZIP: `/kaggle/working/triage_eg_mb1_v02_candidates.zip`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
if not (REPO_DIR / 'src/triage_eg').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'Incomplete repository directory: {REPO_DIR}')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, str(REPO_DIR / 'src'))
print({'resolved_repo': str(REPO_DIR), 'ref': REPO_REF, 'commit': commit})

In [ ]:
DATASET_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
MB1_INPUT = Path(os.environ.get('AIC_MB1_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-mb1-candidates'))
QC_INPUT = Path(os.environ.get('AIC_MB1_QC_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-mb1-ai-annotations'))
RT2_INPUT = Path(os.environ.get('AIC_RT2_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle'))
OUTPUT_ROOT = Path('/kaggle/working/triage_eg_mb1_v02_candidates')
ZIP_PATH = Path('/kaggle/working/triage_eg_mb1_v02_candidates.zip')
print({'dataset': str(DATASET_INPUT), 'mb1': str(MB1_INPUT), 'qc': str(QC_INPUT), 'rt2': str(RT2_INPUT), 'output': str(OUTPUT_ROOT)})

In [ ]:
SEARCH_ROOT = Path('/kaggle/input')
MAX_DEPTH = 6
MAX_DIRECTORIES = 5000

def bounded_directories(root: Path):
    frontier = [(Path(root), 0)]
    visited = 0
    while frontier:
        current, depth = frontier.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError('Kaggle input discovery exceeded directory bound')
        yield current
        if depth < MAX_DEPTH:
            frontier.extend((item, depth + 1) for item in sorted(current.iterdir()) if item.is_dir() and not item.is_symlink())

def resolve_file(root: Path, filename: str, *, required: bool = True):
    direct = root if root.is_file() else root / filename
    if direct.is_file() and direct.name == filename:
        return direct.resolve()
    matches = [directory / filename for directory in bounded_directories(root if root.exists() else SEARCH_ROOT) if (directory / filename).is_file()]
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) == 1:
        return matches[0]
    if not matches and not required:
        return None
    raise RuntimeError(f'Expected exactly one {filename}; found {matches}')

def resolve_dataset(root: Path):
    candidates = [directory for directory in bounded_directories(root if root.exists() else SEARCH_ROOT) if any(directory.glob('Videos_*'))]
    candidates = sorted(set(path.resolve() for path in candidates))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one raw dataset root; found {candidates}')
    return candidates[0]

DATASET_ROOT = resolve_dataset(DATASET_INPUT)
PRIOR_MANIFEST = resolve_file(MB1_INPUT, 'mb1_candidate_manifest.jsonl')
PRIOR_QC = resolve_file(QC_INPUT, 'mb1_candidate_qc.jsonl')
RT2_BENCHMARK = resolve_file(RT2_INPUT, 'rt2_ai_benchmark.jsonl')
PRIOR_SELECTION = resolve_file(MB1_INPUT, 'candidate_selection.json', required=False)
print({'dataset_root': str(DATASET_ROOT), 'prior_manifest': str(PRIOR_MANIFEST), 'prior_qc': str(PRIOR_QC), 'rt2_benchmark': str(RT2_BENCHMARK), 'prior_selection': str(PRIOR_SELECTION) if PRIOR_SELECTION else None})

In [ ]:
from triage_eg.experiments.mb1_v02 import MB1V02Config, preflight_mb1_v02

for path in (OUTPUT_ROOT,):
    if path.exists():
        if path.parent != Path('/kaggle/working'):
            raise RuntimeError(f'Refusing cleanup outside /kaggle/working: {path}')
        shutil.rmtree(path)
ZIP_PATH.unlink(missing_ok=True)
CONFIG = MB1V02Config(dataset_root=DATASET_ROOT, prior_candidate_manifest_path=PRIOR_MANIFEST, prior_candidate_qc_path=PRIOR_QC, rt2_benchmark_path=RT2_BENCHMARK, prior_selection_path=PRIOR_SELECTION, output_root=OUTPUT_ROOT, build_git_commit=commit)
PREFLIGHT = preflight_mb1_v02(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))
assert PREFLIGHT['model_inference_required'] is False
assert PREFLIGHT['previous_semantic_intervals_required'] is False

In [ ]:
from triage_eg.experiments.mb1_v02 import prepare_mb1_v02_candidates

RESULT = prepare_mb1_v02_candidates(CONFIG)
SELECTION = RESULT['selection']
print(json.dumps({key: SELECTION[key] for key in ('source_video_count_considered', 'candidate_proposals_before_filtering', 'candidates_rejected_for_hard_cut_overlap', 'candidates_rejected_for_insufficient_activity', 'candidates_rejected_by_temporal_nms', 'final_candidate_count', 'per_video_candidate_counts', 'hard_cut_overlap_count')}, indent=2))

In [ ]:
rows = [json.loads(line) for line in (OUTPUT_ROOT / 'mb1_v02_candidate_manifest.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
diagnostics = [json.loads(line) for line in (OUTPUT_ROOT / 'mb1_v02_candidate_diagnostics.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == len(diagnostics) == SELECTION['final_candidate_count']
assert all(row['overview_displayed_frames'] == sorted(set(row['overview_displayed_frames'])) for row in rows)
assert all(row['dense_displayed_frames'] == sorted(set(row['dense_displayed_frames'])) for row in rows)
assert all((OUTPUT_ROOT / row['overview_sheet_path']).is_file() and (OUTPUT_ROOT / row['dense_sheet_path']).is_file() for row in rows)
assert SELECTION['hard_cut_overlap_count'] == 0
print('manifest mapping: EXACT')
print('overview sheets:', len(rows), 'dense sheets:', len(rows))
print('MB1_V02_REAL_STATUS = COMPLETE')
print('MB1_V02_AI_QC_STATUS = WAITING_FOR_AI')
print('M3_IMPLEMENTATION_STATUS = NOT_STARTED')

In [ ]:
from IPython.display import Image, display

for row in rows[:3]:
    print(row['candidate_id'], row['video_id'], row['window_start_frame'], row['window_end_frame'])
    display(Image(filename=str(OUTPUT_ROOT / row['overview_sheet_path'])))
    display(Image(filename=str(OUTPUT_ROOT / row['dense_sheet_path'])))

In [ ]:
from zipfile import ZipFile
from triage_eg.experiments.mb1_v02 import create_mb1_v02_bundle

archive = create_mb1_v02_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(archive) as stream:
    members = stream.namelist()
assert not any(name.endswith(('.mp4', '.npy', '.npz', '.pt', '.pth', '.bin')) for name in members)
print('DOWNLOAD ZIP:', archive)
print('size_bytes:', archive.stat().st_size, 'members:', len(members))